# LCE Store Locations - Bronze Layer

Transforms raw LCE store location data into standardized format.

**Input**: Raw LCE locations table (manually uploaded)

**Output**: `{catalog}.{bronze_schema}.lce_locations_mass` - Filtered and standardized MA store locations

## Parameters

In [ ]:
from pyspark.sql import functions as F
from datetime import datetime

# Notebook parameters
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("bronze_schema", "")
dbutils.widgets.text("lce_locations_table", "")
dbutils.widgets.text("state_filter", "MA")

# Extract parameters
catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
lce_locations_table = dbutils.widgets.get("lce_locations_table")
state_filter = dbutils.widgets.get("state_filter")

assert catalog and bronze_schema and lce_locations_table, "Missing required parameters: catalog, bronze_schema, lce_locations_table"

# Define output table
output_table = f"{catalog}.{bronze_schema}.lce_locations_mass"

print(f"Catalog: {catalog}")
print(f"Bronze schema: {bronze_schema}")
print(f"Input table: {lce_locations_table}")
print(f"State filter: {state_filter}")
print(f"Output table: {output_table}")

## Validate Input Table

In [ ]:
# Validate input table exists
try:
    input_df = spark.table(lce_locations_table)
    input_count = input_df.count()
    print(f"✓ Input table found: {lce_locations_table}")
    print(f"  Total records: {input_count:,}")
except Exception as e:
    print(f"\n❌ ERROR: Input table not found: {lce_locations_table}")
    print(f"\nPlease ensure the LCE locations table has been manually uploaded.")
    print(f"\nExpected table: {lce_locations_table}")
    print(f"Required columns: LocationKey, Y_Coordinate_Latitude, X_Coordinate_Longitude, Address1, City, State, Zip, StoreStatus")
    raise RuntimeError(f"Input table not found: {lce_locations_table}") from e

# Validate required columns
required_columns = ['LocationKey', 'Y_Coordinate_Latitude', 'X_Coordinate_Longitude', 
                    'Address1', 'City', 'State', 'Zip', 'StoreStatus']
missing_columns = [col for col in required_columns if col not in input_df.columns]

if missing_columns:
    print(f"\n❌ ERROR: Missing required columns: {missing_columns}")
    print(f"\nAvailable columns: {input_df.columns}")
    raise ValueError(f"Missing required columns: {missing_columns}")

print(f"\n✓ All required columns present")

## Create Standardized LCE Locations Table

In [ ]:
# Create table using SQL CTAS
spark.sql(f"""
CREATE OR REPLACE TABLE {output_table}
COMMENT 'Little Caesars store locations in Massachusetts - filtered for open stores only'
AS
SELECT 
  LocationKey AS location_id,
  concat("Little Caesar's - ", City) AS store_name,
  Y_Coordinate_Latitude AS latitude,
  X_Coordinate_Longitude AS longitude,
  Address1 AS address,
  City AS city,
  State AS state,
  Zip AS zip_code,
  current_timestamp() AS ingestion_timestamp
FROM {lce_locations_table}
WHERE State = '{state_filter}' 
  AND StoreStatus = 'Open Store'
""")

print(f"✓ Created table: {output_table}")

## Validation

In [ ]:
print("=" * 80)
print("LCE LOCATIONS VALIDATION")
print("=" * 80)

# Summary statistics
summary = spark.sql(f"""
    SELECT 
        COUNT(*) as total_stores,
        COUNT(DISTINCT location_id) as unique_locations,
        COUNT(CASE WHEN latitude IS NOT NULL AND longitude IS NOT NULL THEN 1 END) as stores_with_coords,
        COUNT(DISTINCT city) as unique_cities,
        COUNT(DISTINCT state) as unique_states
    FROM {output_table}
""")
display(summary)

# Sample records
print("\nSample LCE store locations:")
display(spark.table(output_table).limit(10))

# Validate all coordinates are present
missing_coords = spark.sql(f"""
    SELECT COUNT(*) as missing_count
    FROM {output_table}
    WHERE latitude IS NULL OR longitude IS NULL
""").collect()[0]['missing_count']

if missing_coords > 0:
    print(f"\n⚠️  WARNING: {missing_coords} stores missing coordinates")
else:
    print(f"\n✓ All stores have valid coordinates")

print("\n" + "=" * 80)
print("VALIDATION COMPLETE")
print("=" * 80)